# 04 — Transect
## Cut the measurement line across the DM band; plot all channels

The transect is drawn **perpendicular to the merger axis**, crossing the midpoint
between DM NW and DM SE. It is the measurement instrument.

This notebook plots every channel along the transect:
- Faraday RM
- Polarised intensity PI
- E-vector PA (EVPA)
- X-ray surface brightness Σ_X
- Lensing convergence κ
- Compton-y

No diagnostic verdict here — that is notebook 05.

---

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../modules')
from constants import RA0, DEC0, DM_NW, DM_SE, GAS_NW, GAS_SE, MERGER_PA_DEG
from transect import transect_endpoints, sample_points

BASE  = Path('/media/rendier/0123-4567/bullet_cluster')
OUT   = Path('../output')
OUT.mkdir(exist_ok=True)

# The transect — perpendicular to merger axis, 10 arcmin, 200 points
T    = transect_endpoints(length_arcmin=10.0)
ra_arr, dec_arr, offset = sample_points(T, n_points=200)

print(f'Transect centre:  RA={T["mid_ra"]:.4f}  Dec={T["mid_dec"]:.4f}')
print(f'PA (transect):    {T["pa_deg"]:.1f}° (merger PA={MERGER_PA_DEG:.1f}°, transect=merger+90°)')
print(f'Length:           {T["length_arcmin"]:.1f}\'')
print(f'Points:           {len(ra_arr)}')
print(f'Start:  RA={T["ra_start"]:.4f}  Dec={T["dec_start"]:.4f}')
print(f'End:    RA={T["ra_end"]:.4f}    Dec={T["dec_end"]:.4f}')

In [ ]:
# Extract RM and PI profiles for both synthetic models
from transect import extract_rm_from_qu_cubes

SYNTH = BASE / 'radio/meerkat/synthetic'
REAL_Q = BASE / 'radio/meerkat/MGCLS_1E0657-558_Q_cube.fits'

profiles = {}

for dm_model in ['wave', 'particle']:
    q_path = SYNTH / f'synthetic_Q_{dm_model}.fits'
    u_path = SYNTH / f'synthetic_U_{dm_model}.fits'
    f_path = SYNTH / f'synthetic_freqs_{dm_model}.dat'
    if not q_path.exists():
        print(f'Synthetic {dm_model} cubes not found — run notebook 03 first')
        continue
    print(f'Extracting RM profile (dm_model={dm_model}) ...')
    rm, pi, evpa = extract_rm_from_qu_cubes(
        str(q_path), str(u_path), str(f_path), ra_arr, dec_arr
    )
    profiles[dm_model] = dict(rm=rm, pi=pi, evpa=evpa)
    print(f'  RM range: [{np.nanmin(rm):.1f}, {np.nanmax(rm):.1f}] rad/m²')
    print(f'  PI range: [{np.nanmin(pi):.4f}, {np.nanmax(pi):.4f}]')

# Real data if available
if REAL_Q.exists():
    real_freq = BASE / 'radio/meerkat/MGCLS_freqs.dat'
    real_u    = BASE / 'radio/meerkat/MGCLS_1E0657-558_U_cube.fits'
    print('Extracting RM profile (REAL MGCLS) ...')
    rm_r, pi_r, evpa_r = extract_rm_from_qu_cubes(
        str(REAL_Q), str(real_u), str(real_freq), ra_arr, dec_arr
    )
    profiles['real'] = dict(rm=rm_r, pi=pi_r, evpa=evpa_r)
    print(f'  RM range (REAL): [{np.nanmin(rm_r):.1f}, {np.nanmax(rm_r):.1f}] rad/m²')

In [ ]:
# Extract X-ray and lensing kappa from viewer PNG layers
from PIL import Image
from transect import _interp2d

LAYERS = BASE / 'viewer' / 'layers'
SIZE, SCALE = 800, 1.0/3600   # 800px, 1"/px

def png_transect(name):
    p = LAYERS / f'{name}.png'
    if not p.exists():
        return None
    img = np.array(Image.open(p).convert('L')).astype(float)
    px  = SIZE/2 - (ra_arr - RA0)  / SCALE
    py  = SIZE/2 + (dec_arr - DEC0) / SCALE
    return _interp2d(img, px, py)

xray  = png_transect('xray_chandra')
kappa = png_transect('lensing_kappa')
sz    = png_transect('sz_planck')

print(f'X-ray profile:  {"present" if xray is not None else "missing"}')
print(f'Kappa profile:  {"present" if kappa is not None else "missing"}')
print(f'SZ profile:     {"present" if sz is not None else "missing"}')

In [ ]:
# Full transect plot — one row per observable
MARKERS = {
    'DM NW':  (DM_NW['ra'],  DM_NW['dec'],  'blue',  '--'),
    'DM SE':  (DM_SE['ra'],  DM_SE['dec'],  'blue',  ':'),
    'Gas NW': (GAS_NW['ra'], GAS_NW['dec'], 'orange', '--'),
    'Gas SE': (GAS_SE['ra'], GAS_SE['dec'], 'orange', ':'),
}

def marker_offsets():
    """Compute transect offsets for each physical landmark."""
    result = {}
    for label, (ra, dec, col, ls) in MARKERS.items():
        dr = (ra  - T['mid_ra'])  * np.cos(np.radians(T['mid_dec'])) * 60
        dd = (dec - T['mid_dec']) * 60
        result[label] = (np.sqrt(dr**2 + dd**2) * np.sign(dr + dd), col, ls)
    return result

moff = marker_offsets()

n_rows = 2 + len(profiles)
fig, axes = plt.subplots(n_rows, 1, figsize=(12, 3*n_rows), sharex=True)
row = 0

def add_markers(ax):
    for label, (xv, col, ls) in moff.items():
        ax.axvline(xv, color=col, ls=ls, lw=0.8, alpha=0.7)
        ax.text(xv, ax.get_ylim()[1]*0.9, label[:4], color=col, fontsize=7, ha='center')

# --- X-ray ---
ax = axes[row]; row += 1
if xray is not None:
    ax.plot(offset, xray, 'r', lw=0.8)
else:
    ax.text(0.5, 0.5, 'X-ray: generate layers first', ha='center', transform=ax.transAxes)
ax.set_ylabel('Σ_X (counts/px)')
ax.set_title('X-ray surface brightness (Chandra 0.5–7 keV)')

# --- Kappa ---
ax = axes[row]; row += 1
if kappa is not None:
    ax.plot(offset, kappa, 'b', lw=0.8)
else:
    ax.text(0.5, 0.5, 'κ: generate layers first', ha='center', transform=ax.transAxes)
ax.set_ylabel('κ (lensing)')
ax.set_title('Gravitational convergence (NFW model, Clowe+2006)')

# --- RM profiles ---
colours = {'wave': 'green', 'particle': 'purple', 'real': 'black'}
for dm_model, prof in profiles.items():
    ax = axes[row]; row += 1
    ax.plot(offset, prof['rm'], color=colours[dm_model], lw=0.8)
    ax.set_ylabel('RM (rad/m²)')
    label = 'REAL MGCLS' if dm_model == 'real' else f'Synthetic ({dm_model} DM)'
    ax.set_title(f'Faraday rotation measure — {label}')
    add_markers(ax)

axes[-1].set_xlabel('Offset along transect (arcmin)')
plt.suptitle('Transect profiles across the DM band (perpendicular to merger axis)', y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'transect_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/transect_profiles.png')